In [1]:
import sys
!uv pip install --python {sys.executable} --reinstall "tqdm==4.67.1" "transformers==4.56.2" "diffusers==0.32.2" "gradio==5.49.1" "huggingface-hub==0.36.2"

Using Python 3.12.12 environment at: /Users/michaelperrine/Documents/Donner/ai-engineer-core-donner/llm_engineering/.venv
Resolved 62 packages in 131ms                                        
Prepared 62 packages in 1ms                                              
Uninstalled 80 packages in 34.21s
Installed 62 packages in 284ms                              
 ~ aiofiles==24.1.0
 ~ annotated-doc==0.0.5
 ~ annotated-types==0.8.0
 - anyio==4.11.0
 ~ anyio==4.15.1
 ~ brotli==1.2.0
 - certifi==2025.10.5
 ~ certifi==2026.7.22
 - charset-normalizer==3.4.4
 ~ charset-normalizer==3.5.1
 ~ click==8.5.0
 ~ diffusers==0.32.2
 - fastapi==0.121.0
 ~ fastapi==0.141.1
 ~ ffmpy==1.0.0
 - filelock==3.20.0
 ~ filelock==4.0.1
 - fsspec==2025.3.0
 + fsspec==2026.9.0
 ~ gradio==5.49.1
 ~ gradio-client==1.13.3
 ~ groovy==0.1.2
 ~ h11==0.16.0
 - hf-xet==1.2.0
 ~ hf-xet==1.6.0
 ~ httpcore==1.0.9
 ~ httpx==0.28.1
 - huggingface-hub==0.36.0
 ~ huggingface-hub==0.36.2
 ~ idna==3.20
 - importlib-metadata==8.7.0
 +

In [2]:
import sys
!uv pip install --python {sys.executable} --reinstall --no-deps "bitsandbytes==0.50.2" "accelerate==1.14.0"

Using Python 3.12.12 environment at: /Users/michaelperrine/Documents/Donner/ai-engineer-core-donner/llm_engineering/.venv
Resolved 2 packages in 45ms                                          
Prepared 2 packages in 0.18ms                                            
Uninstalled 2 packages in 12ms
Installed 2 packages in 9ms                                 
 ~ accelerate==1.14.0
 ~ bitsandbytes==0.50.2


In [3]:
!uv pip install -q --upgrade datasets==3.6.0

In [4]:
# Add code to run Apple mps
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
else: 
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [5]:
#from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM,TextStreamer, BitsAndBytesConfig
import torch
import gc




ValueError: Unable to compare versions for tqdm>=4.27: need=4.27 found=None. This is unusual. Consider reinstalling tqdm.

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoTokenizer

hf_token = getpass("Hugging Face token: ")
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

# Confirm which account owns the token.
print("Authenticated as:", HfApi().whoami(token=hf_token)["name"])

# Check access with a small file before loading the model.
hf_hub_download(
    repo_id=LLAMA,
    filename="config.json",
    token=hf_token,
)
print("Model access confirmed")

tokenizer = AutoTokenizer.from_pretrained(
    LLAMA,
    token=hf_token,
)
tokenizer.pad_token = tokenizer.eos_token

Hugging Face token: ··········
Authenticated as: mestico01
Model access confirmed


In [ ]:
# Instruct models and 1 reasoning model

# Llama 3.1 is larger and you should alreadybe approved
# see here: https://huggingface.co/meta-llama-3.1-8B-Instruct

# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"



In [ ]:
messages = [
    {
        "role": "system",
        "content": "Tell a joke for a room of Data Scientists"
    }
]

In [ ]:
# Quantization Config - This allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages,
                                       return_tensors="pt").to("cuda")


In [ ]:
inputs

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    508,  17907,    220,   2366,     21,    271,  41551,    264,
          22380,    369,    264,   3130,    315,   2956,  57116, 128009]],
       device='cuda:0')

In [ ]:
model = AutoModelForCausalLM.from_pretrained(LLAMA,
                                             device_map="auto",
                                             quantization_config=quant_config,
                                             token=hf_token)

In [ ]:
print(model)
print(type(model))

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), 

In [ ]:
memory = model.get_memory_footprint() /1e6
print(f"Memory footprint: {memory:,.1f} MB")

Memory footprint: 1,012.0 MB


In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), 

In [ ]:
outputs = model.generate(inputs,
                         max_new_tokens = 80)
outputs[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,    508,  17907,    220,   2366,     21,    271,  41551,    264,
         22380,    369,    264,   3130,    315,   2956,  57116, 128009, 128006,
         78191, 128007,    271,   8586,    596,    264,  22380,  41891,    369,
           264,   3130,    315,    828,  14248,   1473,  10445,   1550,    279,
         10550,    733,    311,  15419,   1980,  18433,    433,    574,   8430,
           264,   2697,    330,    359,  52243,      1,    323,   4460,    311,
           330,  31690,      1,   1202,  21958,   2268,   2028,  22380,  11335,
           389,    279,  11156,   3878,   1511,    304,    828,   8198,     11,
          1778,    439,    330,  22090,      1,    320,    695,    705,    330,
           359,  52243,      1,    320,    318,  64185,    477,   7257,    828,
           705,    323,    330,  31690, 

In [ ]:
tokenizer.decode(outputs[0])

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 20 Sep 2026\n\nTell a joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHere\'s a joke tailored for a room of data scientists:\n\nWhy did the dataset go to therapy?\n\nBecause it was feeling a little "unstructured" and needed to "normalize" its emotions!\n\nThis joke plays on the technical terms used in data science, such as "dataset" (data), "unstructured" (imperfect or raw data), and "normalize" (making'

In [ ]:
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def generate(model_name, messages, hf_token, quant=True, max_new_tokens = 80, quant_config=None):
  tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages,
                                            return_tensors="pt",
                                            add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids,
                                     dtype=torch.long,
                                     device = "cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    if quant_config is None:
        raise ValueError("quant_config must be provided when quant=True")
    model = AutoModelForCausalLM.from_pretrained(model_name,
                                                quantization_config=quant_config,
                                                token=hf_token).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model_name, token=hf_token).to("cuda")
  outputs = model.generate(input_ids=input_ids,
                            attention_mask=attention_mask,
                            max_new_tokens=max_new_tokens,
                            streamer=streamer)

In [ ]:
generate(PHI, messages, hf_token=hf_token, quant_config=quant_config)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

<|system|>Tell a joke for a room of Data Scientists<|end|><|assistant|>Sure, here's a joke tailored for data scientists:

Why don't data scientists ever trust atoms?

Because they make up everything!<|end|>


In [ ]:
messages = [
    {
        "role": "user",
        "content": "Tell a light hearted joke for a room of Data Scientists"
    }
]
generate(GEMMA, messages, hf_token=hf_token, quant=False)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<bos><start_of_turn>user
Tell a light hearted joke for a room of Data Scientists<end_of_turn>
<start_of_turn>model
Why don't data scientists ever have a fancy teapot? 

Because they're always running late! 
<end_of_turn>


In [ ]:
generate(QWEN, messages, hf_token=hf_token, quant=False)

NameError: name 'generate' is not defined

In [ ]:
generate(DEEPSEEK, messages, hf_token=hf_token, quant=False)